<a href="https://colab.research.google.com/github/ajeyamk/Data-Science--Cheat-Sheet/blob/master/PS1_Reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this experiment, you will explore the accuracy of sentiment classificaiton using different feature representations of text documents.

First, you will implement `createBasicFeatures`, which creates a sparse matrix representation of a collection of documents. For this exercise, you should have a feature for each word containing at least one alphabetic character. You may use the `numpy` and `sklearn` packages to help with implementing a sparse matrix.

Then, you will implement `createFancyFeatures`, which can specify at any other features you choose to help improve performance on the classification task.

The two code blocks at the end train and evaluate two models—logistic regression with L1 and L2 regularization—using your featurization functions. Besides held-out classification accuracy with 10-fold cross-validation, you will also see the features in each class given high weights by the model.

In [0]:
import json
import re
import requests
import functools 
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate,LeaveOneOut,KFold

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.stem.porter import PorterStemmer

# nltk.download('stopwords')
# nltk.download('wordnet')

In [0]:
class ClassifyReviews():
  """
  Classification class
  """

  def __init__(self):
    self.corpus = self.read_reviews()
    self.corpus_df = pd.DataFrame(self.corpus)
    self.lemmatizer = WordNetLemmatizer()
    self.stemmer = PorterStemmer()
    self.count_vectorizer = CountVectorizer()
    self.tfidf_vectorizer = TfidfVectorizer()

  def read_reviews(self):
    """
    Loads the reviews data 
    """
    raw = requests.get("https://raw.githubusercontent.com/mutherr/CS6120-PS1-data/master/cornell_reviews.json").text.strip()
    corpus = [json.loads(line) for line in raw.split("\n")]
    return corpus

  def handle_stop_words(self,text):
    """
    Handles punctuations and stopwords in english
    """
    stop_words_agg = stopwords.words("english") + list(string.punctuation)
    return [word for  word in text if word not in stop_words_agg]

  def lemmatize_words(self,text, lemmatizer):
    """
    Helper function to lemmatize words
    """
    return [lemmatizer.lemmatize(i) for i in text]

  def stem_words(self, text, stemmer):
    """
    Helper function to stem english words
    """
    return [stemmer.stem(i) for i in text]

  def create_basic_features(self):
    """
    To create basic features
    """
    df = self.corpus_df
    # removes non alpha characters
    df['processed_text'] = df.text.apply(lambda x: [word for word in x.split() if word.isalpha()])
    df['processed_text'] = df.processed_text.apply(lambda x: " ".join(i for i in x))
    sparse_matrix = self.count_vectorizer.fit_transform(df.processed_text.to_list())
    # return sparse matrix, labe class and vocabulary
    return sparse_matrix, df['class'].to_list(), self.count_vectorizer.get_feature_names()

  def create_fancy_features(self, type_eval=None, vectorizer_type='count'):
    """
    Uses count vectorizer
    To create better feature representations that leads to increased accuracy.
      - stop words technique
      - Lemmatization technique
      - Stemming technique
      - N-gram technique
      - Lower casing
    """
    # remove non alphabetic chars
    df = self.corpus_df
    df['processed_text'] = df.text.apply(lambda x: [word for word in x.split() if word.isalpha()])
    df['class'] = df['class'].map({'pos': 1, 'neg': 0})

    # Process for stop words from NLTK package 
    if type_eval == 'stop_words':
      df['processed_text'] = df.processed_text.apply(lambda x: handle_stop_words(x))
      df['processed_text'] = df.processed_text.apply(lambda x: " ".join(i for i in x))
    # Lemmatization technique using word net lemmatizer
    if type_eval == 'lemmatize':
      df['processed_text'] = df.processed_text.apply(lambda x: self.lemmatizer(x, lemmatizer))
      df['processed_text'] = df.processed_text.apply(lambda x: " ".join(i for i in x))
    # Porter stemmer technique
    if type_eval == 'stem':
      df['processed_text'] = df.processed_text.apply(lambda x: self.stemmer(x, stemmer))
      df['processed_text'] = df.processed_text.apply(lambda x: " ".join(i for i in x))
    # N-Gram technique
    if type_eval == 'ngram_2':
      df['processed_text'] = df.processed_text.apply(lambda x: " ".join(i for i in x))
    # convert to lower case technique
    if type_eval == 'lower_case':
      df['processed_text'] = df.processed_text.apply(lambda x: " ".join(i for i in x))
      df['processed_text'] = df['processed_text'].str.lower()
    
    if vectorizer_type == 'count':
      vectorizer = CountVectorizer(analyzer='word', ngram_range=(1, 2)) if type_eval == 'ngram_2' else self.count_vectorizer
    else:
      vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1, 2)) if type_eval == 'ngram_2' else self.tfidf_vectorizer

    spr_matrix = vectorizer.fit_transform(df.processed_text.to_list())
    # return sparse matrix, labe class and vocabulary
    return spr_matrix, df['class'].to_list(), vectorizer.get_feature_names()

  def evaluate_model(self, X,y,vocab,penalty="l1"):
    #create and fit the model
    model = LogisticRegression(penalty=penalty,solver="liblinear")
    results = cross_validate(model,X,y,cv=KFold(n_splits=10, shuffle=True, random_state=1))
    
    #determine the average accuracy
    scores = results["test_score"]
    avg_score = sum(scores)/len(scores)
    
    #determine the most informative features
    # this requires us to fit the model to everything, because we need a
    # single model to draw coefficients from, rather than 26
    model.fit(X,y)
    class0_weight_sorted = model.coef_[0, :].argsort()
    class1_weight_sorted = (-model.coef_[0, :]).argsort()

    termsToTake = 20
    class0_indicators = [vocab[i] for i in class0_weight_sorted[:termsToTake]]
    class1_indicators = [vocab[i] for i in class1_weight_sorted[:termsToTake]]

    if model.classes_[0] == "pos":
      return avg_score,class0_indicators,class1_indicators
    else:
      return avg_score,class1_indicators,class0_indicators

  def run_evaluation(self, X,y,vocab):
    print("----------L1 Norm-----------")
    avg_score,pos_indicators,neg_indicators = self.evaluate_model(X,y,vocab,"l1")
    print("The model's average accuracy is %f"%avg_score)
    print("The most informative terms for pos are: %s"%pos_indicators)
    print("The most informative terms for neg are: %s"%neg_indicators)
    #this call will fit a model with L2 normalization
    print("----------L2 Norm-----------")
    avg_score,pos_indicators,neg_indicators = self.evaluate_model(X,y,vocab,"l2")
    print("The model's average accuracy is %f"%avg_score)
    print("The most informative terms for pos are: %s"%pos_indicators)
    print("The most informative terms for neg are: %s"%neg_indicators)


In [0]:
classifier = ClassifyReviews()

Run the following to train and evaluate two models using basic features:

In [0]:
X,y,vocab = classifier.create_basic_features()
classifier.run_evaluation(X, y, vocab)

----------L1 Norm-----------
The model's average accuracy is 0.824500
The most informative terms for pos are: ['flaws', 'fantastic', 'excellent', 'terrific', 'memorable', 'command', 'using', 'edge', 'musicians', 'liar', 'sherri', 'perfectly', 'easily', 'jackie', 'fun', 'allows', 'gas', 'follows', 'enjoyable', 'masterpiece']
The most informative terms for neg are: ['waste', 'tedious', 'mess', 'awful', 'lame', 'ridiculous', 'worst', 'unfortunately', 'write', 'cheap', 'boring', 'bad', 'superior', 'headed', 'terrible', 'designed', 'jesse', 'nothing', 'looks', 'jakob']
----------L2 Norm-----------
The model's average accuracy is 0.837500
The most informative terms for pos are: ['fun', 'back', 'great', 'excellent', 'seen', 'yet', 'perfectly', 'overall', 'quite', 'well', 'jackie', 'memorable', 'terrific', 'pulp', 'job', 'american', 'follows', 'true', 'hilarious', 'bit']
The most informative terms for neg are: ['bad', 'unfortunately', 'worst', 'nothing', 'waste', 'boring', 'only', 'awful', 'lo

Run the following to train and evaluate two models using extended features:

In [0]:
# We'll choose ngram that combines unigrams and bigrams 
X,y,vocab = classifier.create_fancy_features(type_eval='ngram_2', vectorizer_type='count')
classifier.run_evaluation(X, y, vocab)

----------L1 Norm-----------
The model's average accuracy is 0.836500
The most informative terms for pos are: ['even if', 'flaws', 'as much', 'perfectly', 'terrific', 'masterpiece', 'excellent', 'fantastic', 'on screen', 'follows', 'overall', 'view', 'her husband', 'memorable', 'gas', 'the true', 'due to', 'great', 'fun', 'works']
The most informative terms for neg are: ['ridiculous', 'waste', 'awful', 'mess', 'unfortunately', 'worst', 'lame', 'cheap', 'tedious', 'should have', 'bad', 'write', 'terrible', 'flat', 'headed', 'poor', 'boring', 'designed', 'metro', 'to show']
----------L2 Norm-----------
The model's average accuracy is 0.854000
The most informative terms for pos are: ['great', 'fun', 'seen', 'well', 'back', 'very', 'also', 'quite', 'excellent', 'yet', 'job', 'many', 'perfectly', 'jackie', 'people', 'life', 'most', 'hilarious', 'you', 'while']
The most informative terms for neg are: ['bad', 'only', 'unfortunately', 'nothing', 'worst', 'boring', 'any', 'waste', 'script', 'pl